# 🚌 Trip Visualizer & Stop Editor

This notebook lets you:
1. **Browse** all available trips with their route names
2. **Visualize** a trip's stop sequence on a table and interactive map
3. **Add new stops** to a trip and save changes back to `stop_times.txt`

## 1. Import Libraries & Load GTFS Data

In [8]:
import pandas as pd
import folium
from IPython.display import display, HTML
from pathlib import Path

# Path to GTFS data
GTFS_DIR = Path("gtfsAlex")

# Load GTFS files
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
routes = pd.read_csv(GTFS_DIR / "routes.txt")
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")

print(f"Loaded: {len(stop_times)} stop_times, {len(stops)} stops, {len(trips)} trips, {len(routes)} routes, {len(shapes)} shape points")

Loaded: 2548 stop_times, 441 stops, 192 trips, 104 routes, 29359 shape points


## 2. Browse All Trips
Select a `trip_id` from the table below to use in the next cells.

In [9]:
# Merge trips with routes to show route names
trip_info = trips.merge(routes[["route_id", "route_long_name"]], on="route_id", how="left")
trip_summary = trip_info[["trip_id", "route_long_name", "trip_headsign", "direction_id"]].reset_index(drop=True)
trip_summary.index += 1
trip_summary.index.name = "#"
display(trip_summary)

,trip_id,route_long_name,trip_headsign,direction_id
#,,,,
1,OvxARhItfijPV-_xuvfnZ-07:00:00,Asafra - Sidi Bishr,Asafra,1
2,zivMUtHDofqSlzEUoK8F0-07:00:00,Asafra - El-Mawqaf El-Geded,Asafra,1
3,Wq0wtD2-ddsT-JtVfCc3g-07:00:00,Asafra - Hadra,Asafra,0
4,OI7bd_u7laDmk8SxgppK0-07:00:00,Asafra - El-Mansheya,Asafra,1
5,I2ar40PiiuS-HMI9_KCLc-07:00:00,Asafra - El-Sa'ah (Clock Square),Asafra,1
...,...,...,...,...
188,vGWP0NAWQ2WwsR_DTpmn9-07:00:00,Bahary (Ras Al-Tin) - Street 45,Bahary (Ras Al-Tin),1
189,RmCIDxJQKwF7GOCKKX3Oh-07:00:00,Abu Qir - Bahary (Ras Al-Tin),Abu Qir,0
190,LYcY_J8e-Dijvjm6uZhgZ-07:00:00,Abu Qir - Bahary (Ras Al-Tin),Abu Qir,0


## 3. Visualize a Trip's Stop Sequence
Set `TRIP_ID` below to the trip you want to inspect.

In [10]:
############################
# ✏️ SET YOUR TRIP ID HERE #
############################
TRIP_ID = "FUo5FExiKwUTpyTUJYA7R-07:00:00"

def get_trip_stops(trip_id):
    """Get the stop sequence for a trip, merged with stop details."""
    trip_st = stop_times[stop_times["trip_id"] == trip_id].copy()
    if trip_st.empty:
        print(f"⚠️ No stop_times found for trip_id: {trip_id}")
        return pd.DataFrame()
    trip_st = trip_st.merge(stops, on="stop_id", how="left")
    trip_st = trip_st.sort_values("stop_sequence").reset_index(drop=True)
    return trip_st

# Get and display stop sequence
trip_stops = get_trip_stops(TRIP_ID)
if not trip_stops.empty:
    print(f"Trip: {TRIP_ID}")
    trip_row = trips[trips["trip_id"] == TRIP_ID]
    if not trip_row.empty:
        route = routes[routes["route_id"] == trip_row.iloc[0]["route_id"]]
        if not route.empty:
            print(f"Route: {route.iloc[0]['route_long_name']}")
        print(f"Headsign: {trip_row.iloc[0]['trip_headsign']}")
    print(f"Stops: {len(trip_stops)}\n")
    display(trip_stops[["stop_sequence", "stop_id", "stop_name", "stop_name_ar", "arrival_time", "departure_time"]])

Trip: FUo5FExiKwUTpyTUJYA7R-07:00:00
Route: Asafra - Train Station (El-Shohada Square)
Headsign: Train Station (El-Shohada Square)
Stops: 13



,stop_sequence,stop_id,stop_name,stop_name_ar,arrival_time,departure_time
0,1,98,Asafra Station,محطة عصافرة,07:00:00,07:00:15
1,2,45,Street 45 - Miami Bridge,شارع 45 - جسر ميامي,07:01:27,07:01:42
2,3,153,Gehan Square,ميدان جيهان,07:05:52,07:06:07
3,4,255,Tamween Montazah,مكاتب تموين المنتزه,07:09:01,07:09:16
4,5,263,Victoria College,كلية فيكتوريا,07:18:42,07:18:57
5,6,284,Jewelery Museum,متحف المجوهرات,07:27:24,07:27:39
6,7,291,Al Wezara Tram,محطة ترام الوزارة,07:34:19,07:34:34
7,8,341,Mansour Opel Car Showroom,معرض سيارات منصور أوبل,07:45:08,07:45:23
8,9,324,Sidi Gaber Station,محطة سيدى جابر,07:48:46,07:49:01
9,10,414,Misr Gas Station - Ibrahimeya,محطة غاز مصر - إبراهيمية,07:54:51,07:55:06


## 4. Plot Trip on Map
Interactive map showing stops connected in order. 🟢 = first stop, 🔴 = last stop.

In [12]:
def plot_trip_map(trip_stops_df, trip_id=None):
    """Plot trip stops on a folium map with numbered markers, shape polyline, and all stops faded."""
    if trip_stops_df.empty:
        print("No stops to plot.")
        return None

    center = [trip_stops_df["stop_lat"].mean(), trip_stops_df["stop_lon"].mean()]
    m = folium.Map(location=center, zoom_start=13)

    # --- Layer: All stops (faded/shaded) ---
    all_stops_layer = folium.FeatureGroup(name="All Stops (faded)", show=True)
    trip_stop_ids = set(trip_stops_df["stop_id"].values)
    for _, s in stops.iterrows():
        if s["stop_id"] in trip_stop_ids:
            continue  # skip trip stops, they'll be drawn on top
        folium.CircleMarker(
            location=[s["stop_lat"], s["stop_lon"]],
            radius=5,
            color="orange",
            fill=True,
            fill_color="orange",
            fill_opacity=0.8,
            opacity=0.8,
            popup=f"<b>{s['stop_name']}</b><br>{s.get('stop_name_ar', '')}<br>ID: {s['stop_id']}",
            tooltip=f"{s['stop_name']} (id:{s['stop_id']})"
        ).add_to(all_stops_layer)
    all_stops_layer.add_to(m)

    # --- Layer: Trip shape from shapes.txt ---
    if trip_id is not None:
        trip_row = trips[trips["trip_id"] == trip_id]
        if not trip_row.empty:
            shape_id = trip_row.iloc[0].get("shape_id")
            if pd.notna(shape_id):
                shape_pts = shapes[shapes["shape_id"] == shape_id].sort_values("shape_pt_sequence")
                if not shape_pts.empty:
                    shape_layer = folium.FeatureGroup(name="Trip Shape", show=True)
                    shape_coords = shape_pts[["shape_pt_lat", "shape_pt_lon"]].values.tolist()
                    folium.PolyLine(shape_coords, color="purple", weight=4, opacity=0.6, dash_array="8").add_to(shape_layer)
                    # Add clickable shape points
                    for _, pt in shape_pts.iterrows():
                        folium.CircleMarker(
                            location=[pt["shape_pt_lat"], pt["shape_pt_lon"]],
                            radius=3,
                            color="purple",
                            fill=True,
                            fill_color="purple",
                            fill_opacity=0.5,
                            opacity=0.5,
                            popup=f"<b>Shape: {shape_id}</b><br>Seq: {int(pt['shape_pt_sequence'])}<br>Lat: {pt['shape_pt_lat']}<br>Lon: {pt['shape_pt_lon']}"
                        ).add_to(shape_layer)
                    shape_layer.add_to(m)

    # --- Layer: Trip stop connections ---
    trip_layer = folium.FeatureGroup(name="Trip Stops", show=True)
    coords = trip_stops_df[["stop_lat", "stop_lon"]].values.tolist()
    folium.PolyLine(coords, color="blue", weight=3, opacity=0.7).add_to(trip_layer)

    # Add numbered markers for trip stops
    for _, row in trip_stops_df.iterrows():
        seq = int(row["stop_sequence"])
        is_first = seq == trip_stops_df["stop_sequence"].min()
        is_last = seq == trip_stops_df["stop_sequence"].max()

        if is_first:
            color = "green"
        elif is_last:
            color = "red"
        else:
            color = "blue"

        folium.Marker(
            location=[row["stop_lat"], row["stop_lon"]],
            popup=f"<b>#{seq}</b> {row['stop_name']}<br>{row.get('stop_name_ar', '')}<br>Arr: {row['arrival_time']}<br>Dep: {row['departure_time']}",
            tooltip=f"#{seq} {row['stop_name']}",
            icon=folium.DivIcon(html=f'<div style="font-size:10px;color:white;background:{color};border-radius:50%;width:22px;height:22px;text-align:center;line-height:22px;font-weight:bold;border:1px solid #333">{seq}</div>')
        ).add_to(trip_layer)
    trip_layer.add_to(m)

    # Fit bounds & add layer control
    m.fit_bounds([[trip_stops_df["stop_lat"].min(), trip_stops_df["stop_lon"].min()],
                  [trip_stops_df["stop_lat"].max(), trip_stops_df["stop_lon"].max()]])
    folium.LayerControl().add_to(m)
    return m

trip_map = plot_trip_map(trip_stops, trip_id=TRIP_ID)
trip_map.save("trip_map.html")

## 5. Add a New Stop to the Trip
Fill in the parameters below. The new stop will be inserted after the specified `INSERT_AFTER_SEQUENCE` position, and all subsequent stops will be re-numbered.

**Available stops** — run the cell below to search for a stop by name.

In [ ]:
# Search for a stop by name (English or Arabic) — change the query to filter
SEARCH_QUERY = ""  # e.g. "Asafra" or "المنشية" — leave empty to show all stops
filtered = stops[stops["stop_name"].str.contains(SEARCH_QUERY, case=False, na=False) | 
                 stops["stop_name_ar"].str.contains(SEARCH_QUERY, case=False, na=False)] if SEARCH_QUERY else stops
display(filtered[["stop_id", "stop_name", "stop_name_ar", "stop_lat", "stop_lon"]])

In [ ]:
####################################
# ✏️ CONFIGURE THE NEW STOP HERE  #
####################################
NEW_STOP_ID = 6               # stop_id from stops.txt
INSERT_AFTER_SEQUENCE = 3     # insert AFTER this stop_sequence number
NEW_ARRIVAL_TIME = "07:08:00" # arrival time (HH:MM:SS)
NEW_DEPARTURE_TIME = "07:08:15" # departure time (HH:MM:SS)

# --- Validation ---
stop_info = stops[stops["stop_id"] == NEW_STOP_ID]
if stop_info.empty:
    raise ValueError(f"stop_id {NEW_STOP_ID} not found in stops.txt!")

trip_st = stop_times[stop_times["trip_id"] == TRIP_ID].copy()
if trip_st.empty:
    raise ValueError(f"trip_id {TRIP_ID} not found in stop_times!")

max_seq = trip_st["stop_sequence"].max()
if INSERT_AFTER_SEQUENCE < 0 or INSERT_AFTER_SEQUENCE > max_seq:
    raise ValueError(f"INSERT_AFTER_SEQUENCE must be between 0 and {max_seq}")

print(f"✅ Will insert stop '{stop_info.iloc[0]['stop_name']}' (id={NEW_STOP_ID}) after sequence #{INSERT_AFTER_SEQUENCE} in trip {TRIP_ID}")
print(f"   Arrival: {NEW_ARRIVAL_TIME}, Departure: {NEW_DEPARTURE_TIME}")

In [ ]:
# --- Insert the new stop ---

# Bump stop_sequence for all stops AFTER the insertion point (in global stop_times)
mask = (stop_times["trip_id"] == TRIP_ID) & (stop_times["stop_sequence"] > INSERT_AFTER_SEQUENCE)
stop_times.loc[mask, "stop_sequence"] += 1

# Create the new row
new_row = pd.DataFrame([{
    "trip_id": TRIP_ID,
    "stop_id": NEW_STOP_ID,
    "stop_sequence": INSERT_AFTER_SEQUENCE + 1,
    "arrival_time": NEW_ARRIVAL_TIME,
    "departure_time": NEW_DEPARTURE_TIME,
    "timepoint": 0
}])

# Append and re-sort
stop_times = pd.concat([stop_times, new_row], ignore_index=True)
stop_times = stop_times.sort_values(["trip_id", "stop_sequence"]).reset_index(drop=True)

print(f"✅ Stop inserted! Trip now has {len(stop_times[stop_times['trip_id'] == TRIP_ID])} stops.")

# Show updated sequence
updated_trip_stops = get_trip_stops(TRIP_ID)
display(updated_trip_stops[["stop_sequence", "stop_id", "stop_name", "stop_name_ar", "arrival_time", "departure_time"]])

## 6. Re-visualize Updated Trip on Map

In [ ]:
# Plot updated trip map
updated_map = plot_trip_map(updated_trip_stops, trip_id=TRIP_ID)
updated_map

## 7. Save Changes to `stop_times.txt`
Run the cell below to write the updated stop_times back to disk. **This overwrites the original file.**

In [ ]:
# Save updated stop_times back to file
output_path = GTFS_DIR / "stop_times.txt"
stop_times.to_csv(output_path, index=False)
print(f"✅ Saved {len(stop_times)} records to {output_path}")